# Notebook 11 — Dimensionality Reduction

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
ref = pd.Timestamp("2024-06-30")
customers["tenure_days"] = (ref - customers["signup_date"]).dt.days
customers["total_charges"] = customers["total_charges"].fillna(customers["monthly_charges"])
customers["log_total_charges"] = np.log1p(customers["total_charges"])
customers["charge_per_tenure_month"] = customers["monthly_charges"] / customers["tenure_months"].replace(0,1)
customers["churn_binary"] = (customers["churn"]=="Yes").astype(int)

# A block of correlated numeric features (typical scenario where PCA helps)
num_features = ["monthly_charges","tenure_months","tenure_days","total_charges",
                 "log_total_charges","charge_per_tenure_month"]
X = customers[num_features].fillna(0)
X.corr().round(2)

## 1. What is Dimensionality, and the Curse of Dimensionality?

**Dimensionality** = the number of features describing each observation. As
dimensionality grows:
- Data becomes increasingly **sparse** in the feature space — the volume of the space
  grows exponentially, but the number of observations doesn't, so points end up far
  apart from each other even in a "dense" dataset
- Distance-based methods (KNN, clustering) become less meaningful, since all points
  start looking equally "far" from each other
- Models need exponentially more data to generalize well — this is the **Curse of
  Dimensionality**
- Overfitting risk rises, since high-dimensional models can memorize noise more easily

This dataset shows a milder, very common version of the problem: `tenure_months`,
`tenure_days`, `total_charges`, and `log_total_charges` are **highly correlated with
each other** (as the correlation matrix above confirms) — meaning much of that
"dimensionality" is redundant, not new information.

## 2. Why Reduce Dimensions?

- Removes redundancy between correlated features (like our tenure/charges block)
- Speeds up training and reduces overfitting risk
- Enables visualization of high-dimensional data in 2D/3D
- Can improve performance of distance-sensitive algorithms

## 3. PCA (Principal Component Analysis)

PCA finds new axes (**principal components**) that are linear combinations of the
original features, ordered by how much variance in the data they explain. The first
component captures the most variance, the second captures the most *remaining*
variance orthogonal to the first, and so on.

In [ ]:
X_scaled = StandardScaler().fit_transform(X)  # PCA requires scaled features
pca = PCA()
pca.fit(X_scaled)

explained_var = pca.explained_variance_ratio_
cum_var = np.cumsum(explained_var)

for i, (v, cv) in enumerate(zip(explained_var, cum_var), start=1):
    print(f"PC{i}: explains {v*100:5.1f}% of variance | cumulative {cv*100:5.1f}%")

## 4. Explained Variance & Choosing Number of Components

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(range(1, len(cum_var)+1), cum_var, marker="o", color="#4C72B0")
plt.axhline(0.9, color="red", linestyle="--", label="90% variance threshold")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA — Explained Variance vs Number of Components")
plt.legend()
plt.tight_layout()
plt.show()

As expected given the correlation matrix, the first **2 components** already capture
the vast majority of variance — confirming that our 6 tenure/charge-related columns
contain roughly 2 independent dimensions of real information.

## 5. Eigenvectors & Eigenvalues (Under the Hood)

Each principal component is an **eigenvector** of the feature covariance matrix; its
corresponding **eigenvalue** is proportional to the variance it explains. PCA is,
mathematically, just an eigendecomposition of the covariance matrix, re-expressed as
new axes.

In [ ]:
pca_2 = PCA(n_components=2)
components = pca_2.fit_transform(X_scaled)

loadings = pd.DataFrame(pca_2.components_.T, index=num_features, columns=["PC1","PC2"])
loadings

**Interpretation:** the loadings show which original features each principal
component is mostly built from. High-magnitude loadings on PC1 for `total_charges`,
`tenure_months`, and `tenure_days` confirm PC1 essentially represents "customer
tenure/value", exactly as we'd expect from the correlation structure.

## 6. PCA Visualization

In [ ]:
pca_df = pd.DataFrame(components, columns=["PC1","PC2"])
pca_df["churn"] = customers["churn"].values

fig, ax = plt.subplots(figsize=(6,5))
for label, color in [("No","#4C72B0"), ("Yes","#C44E52")]:
    subset = pca_df[pca_df["churn"]==label]
    ax.scatter(subset["PC1"], subset["PC2"], s=8, alpha=0.4, label=f"Churn={label}", color=color)
ax.set_xlabel("PC1 (tenure/value axis)")
ax.set_ylabel("PC2")
ax.legend()
ax.set_title("PCA Projection Colored by Churn")
plt.tight_layout()
plt.show()

## Advantages of PCA
- Removes redundancy from correlated numeric features
- Speeds up downstream training, reduces overfitting risk
- Useful for visualization and exploratory analysis

## Limitations of PCA
- Components are **linear combinations** — loses direct interpretability
  ("PC1" is meaningless to a business stakeholder, unlike `tenure_months`)
- Only captures **linear** structure; non-linear relationships need other techniques
  (e.g. t-SNE, UMAP, autoencoders)
- Sensitive to feature scaling — must standardize first
- Can discard components that, despite low variance, are still predictive of the
  target (PCA is unsupervised — it doesn't know about `churn` at all)

## When to Use PCA
- Many numeric features are highly correlated (as demonstrated here)
- You need to visualize high-dimensional data
- You're using a distance-based or scale-sensitive model with too many raw features

## When NOT to Use PCA
- Interpretability is a hard business requirement (e.g. regulated credit models that
  must explain individual decisions)
- Features are mostly categorical (PCA is designed for continuous numeric data)
- The dataset is already low-dimensional and features aren't strongly correlated —
  there's nothing redundant to compress
- Tree-based models are being used — they already handle correlated/high-dimensional
  features reasonably well natively, so PCA's benefit is smaller